# Stage R&D: Fast In-Memory Slice Pipeline

Demonstrates the in-memory fast path: PFF → calibrated L1 → features → cloud scores — all in RAM, no L0/L1 Zarr materialization on disk.

**When to use**: calibration R&D, ML inference on small time windows, rapid parameter sweeps. Results are scientifically *approximate* when `decimate > 1` (frames are subsampled uniformly; `n_stack` frame groups become non-contiguous in real time). Use the full pipeline (`recipe_driver.run_pipeline`) for production-quality calibration.

## Pipeline
```
PFF .pffd (on disk)
    ↓  open_pff_product  (mmap, zero-copy read)
PFFSequence  [slice: start_ns → stop_ns, step=decimate]
    ↓  slice_to_l1  (in-memory: calibrate_img → median-subtracted)
L1 xr.Dataset  (RAM only)
    ↓  slice_to_l2_cloud  (extract_cloud_features → CNN inference)
L2 xr.Dataset  (cloud_score, cloud_label)
```

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from panoseti_analysis.adapters.slice_driver import slice_to_l1, slice_to_l2_cloud
from panoseti_analysis.algorithms.cloud_detector import CloudDetectionV2
from panoseti_analysis.config.models import CloudInferParams
from panoseti_analysis.io.bench import BenchResult, stage_timer, summarize
from panoseti_analysis.io.models import load_classifier
from panoseti_analysis.io.pff import open_pff_product
from panoseti_analysis.paths import REPO_ROOT

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

## §0 Configuration

In [ ]:
# ── Observation directory ──────────────────────────────────────────────────────
OBS_DIR = Path("/path/to/obs.pffd")  # replace with a real .pffd observation dir
DP = "dp_img16.bpp_2.module_1"  # replace with actual data product (run list_products below)
MODULE = 1

# ── Slice parameters ──────────────────────────────────────────────────────────
# Use None for start/stop to process the whole file; or set to a sub-window.
# Times in nanoseconds (int64). None = use frame_range instead.
START_NS = None  # e.g. int(seq.timestamps()[0])
STOP_NS = None  # e.g. int(seq.timestamps()[3600])  # ~1 hr of 100µs frames
START_FRAME = 0
STOP_FRAME = 10_000  # number of frames to slice (for quick testing)

DECIMATE = 10  # read 1 frame per DECIMATE (10x reduction for R&D)

# ── Recipe (calibration) ──────────────────────────────────────────────────────
RECIPE_PATH = REPO_ROOT / "recipes/ml/cloud_v1.yml"

# ── Cloud detection model ─────────────────────────────────────────────────────
MODEL_PT = REPO_ROOT / "ml/cloud-detection/models/cloud_detector_v2_legacy.pt"
INFER_PARAMS = CloudInferParams(cadence_s=10.0, window_s=60.0, n_stack=10)

## §1 Inspect the PFF sequence

In [ ]:
seq = open_pff_product(OBS_DIR, DP, MODULE)
T = len(seq)
ts = seq.timestamps()  # int64 ns

print(f"Data product : {DP}")
print(f"Total frames : {T:,}")
print(f"Frame config : {seq.frame_config}")
print(f"Duration     : {(ts[-1] - ts[0]) / 1e9:.1f} s  ({(ts[-1] - ts[0]) / 3600e9:.2f} h)")
print(f"Files        : {len(seq.file_paths)}")
print()
print("Available products (if you need to find the right DP name):")
from pypff import PanosetiRun

run = PanosetiRun(str(OBS_DIR))
for p in run.list_products():
    print(f"  {p}")

## §2 Slice to L1 (in-memory, decimated)

In [ ]:
results: list[BenchResult] = []

with stage_timer("slice_to_l1", bytes_in=0) as r_l1:
    ds_l1 = slice_to_l1(
        seq,
        frame_range=(START_FRAME, STOP_FRAME) if START_NS is None else None,
        time_range=(START_NS, STOP_NS) if START_NS is not None else None,
        decimate=DECIMATE,
        recipe=RECIPE_PATH,
        fail_on_suspect=False,
    )
results.append(r_l1)

T_l1 = ds_l1.sizes["time"]
print(
    f"L1 frames    : {T_l1:,}  (from {STOP_FRAME - START_FRAME} input frames, decimate={DECIMATE})"
)
print(f"L1 vars      : {list(ds_l1.data_vars)}")
print(
    f"Time range   : {(ds_l1['unix_t_ns'].values[-1] - ds_l1['unix_t_ns'].values[0]) / 1e9:.1f} s"
)
print(f"Elapsed      : {r_l1.seconds:.2f} s")

## §3 Visualise L1

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Sample frame
med = ds_l1["median_subtracted"].values
sample_frame = med[T_l1 // 2]
im = axes[0].imshow(sample_frame, cmap="RdBu_r", vmin=-5, vmax=5, interpolation="nearest")
axes[0].set_title(f"median_subtracted  frame {T_l1 // 2}")
axes[0].axis("off")
plt.colorbar(im, ax=axes[0], fraction=0.046)

# Time-averaged signal
axes[1].imshow(np.mean(med, axis=0), cmap="viridis", interpolation="nearest")
axes[1].set_title("Mean over time")
axes[1].axis("off")

# Pixel RMS time series (spatial mean of abs values)
ts_s = (ds_l1["unix_t_ns"].values - ds_l1["unix_t_ns"].values[0]) / 1e9
rms = np.mean(np.abs(med), axis=(1, 2))
axes[2].plot(ts_s, rms, lw=0.8)
axes[2].set_xlabel("t (s)")
axes[2].set_ylabel("|median_subtracted| mean")
axes[2].set_title("Spatial mean |signal| vs time")

plt.tight_layout()
plt.show()

## §4 Slice to L2 (cloud detection)

In [ ]:
if not MODEL_PT.exists():
    print(f"Model not found: {MODEL_PT}")
    print("Run notebook 02 first to train the model, or point MODEL_PT to an existing .pt file.")
else:
    model, bundle = load_classifier(MODEL_PT, model=CloudDetectionV2())
    model.eval()
    print(f"Loaded model: {bundle.model_name} v{bundle.model_version}")

    with stage_timer("slice_to_l2_cloud", bytes_in=0) as r_l2:
        ds_l2 = slice_to_l2_cloud(
            seq,
            model,
            frame_range=(START_FRAME, STOP_FRAME) if START_NS is None else None,
            time_range=(START_NS, STOP_NS) if START_NS is not None else None,
            decimate=DECIMATE,
            recipe=RECIPE_PATH,
            infer_params=INFER_PARAMS,
        )
    results.append(r_l2)

    N_l2 = ds_l2.sizes["T_l2"]
    print(f"L2 windows   : {N_l2:,}")
    print(
        f"Cloudy       : {int(ds_l2['cloud_label'].sum())}  ({float(ds_l2['cloud_label'].mean()) * 100:.1f}%)"
    )
    print(f"Elapsed      : {r_l2.seconds:.2f} s")

## §5 Cloud score time series

In [ ]:
if MODEL_PT.exists():
    t_l2_s = (ds_l2["unix_t_ns"].values - ds_l2["unix_t_ns"].values[0]) / 1e9

    fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
    axes[0].plot(t_l2_s, ds_l2["cloud_score"].values, lw=0.8, color="steelblue")
    axes[0].axhline(
        INFER_PARAMS.threshold,
        color="tomato",
        linestyle="--",
        lw=0.8,
        label=f"threshold={INFER_PARAMS.threshold}",
    )
    axes[0].set_ylabel("P(cloudy)")
    axes[0].set_title("Cloud detection score (in-memory slice pipeline)")
    axes[0].legend(fontsize=8)

    axes[1].fill_between(
        t_l2_s, ds_l2["cloud_label"].values.astype(float), alpha=0.6, color="tomato", label="cloudy"
    )
    axes[1].set_xlabel("t (s)")
    axes[1].set_ylabel("Cloud label")
    axes[1].set_ylim(-0.1, 1.1)

    plt.tight_layout()
    plt.show()

## §6 Timing summary

In [ ]:
print(summarize(results))
print()
print("Decimation approximation note:")
print(f"  With decimate={DECIMATE}, frames are spaced {DECIMATE} × cadence apart.")
print("  extract_cloud_features stacks n_stack contiguous (decimated) frames per window.")
print("  This is sufficient for R&D but not production calibration.")